# 🎓 AI Tutor Agent — LangGraph + Groq + Gradio

> **Powered by:** LangGraph · Groq `llama-3.1-8b-instant` · Serper Web Search · Gradio UI

**Use Cases:** 🏫 Schools · 💻 EdTech Platforms · 🏢 Corporate Learning

---

### 🏗️ Agent Architecture

```
┌─────────────┐     ┌──────────────────┐     ┌──────────┐
│   Learner   │────▶│  Learner Profiler│────▶│  Lesson  │
│   Input     │     │  (Level + Style) │     │  Planner │
└─────────────┘     └──────────────────┘     └────┬─────┘
                                                   │
                    ┌──────────────────────────────▼──────────────────┐
                    │              TEACHING NODE                      │
                    │    (Adaptive content + Learning style)          │
                    └──┬──────────────┬──────────────┬────────────────┘
                       │              │              │
                  ┌────▼────┐  ┌─────▼─────┐  ┌────▼──────┐
                  │  Quiz   │  │  Doubt    │  │  Web      │
                  │ Generator│ │  Resolver │  │  Search   │
                  └────┬────┘  └─────┬─────┘  └───────────┘
                       │              │
                  ┌────▼────┐  ┌─────▼─────────────────┐
                  │Feedback │  │  Adaptive Engine       │
                  │ Engine  │  │  (Level Adjustment)    │
                  └─────────┘  └────────────────────────┘
```

---

## 📦 Cell 1 — Install Dependencies

In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║           INSTALL ALL REQUIRED PACKAGES                  ║
# ╚══════════════════════════════════════════════════════════╝

!pip install -q langgraph langchain langchain-groq langchain-community
!pip install -q gradio
!pip install -q requests pydantic typing_extensions

print('✅ All packages installed successfully!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
✅ All packages installed successfully!


## 📚 Cell 2 — Import Libraries

In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║                  IMPORT LIBRARIES                        ║
# ╚══════════════════════════════════════════════════════════╝

import os
import json
import re
import random
import requests
import gradio as gr
from typing import TypedDict, Annotated, List, Optional
from datetime import datetime

# ── LangGraph + LangChain ──
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

print('✅ All libraries imported successfully!')
print(f'  Gradio version : {gr.__version__}')

✅ All libraries imported successfully!
  Gradio version : 5.50.0


## 🔑 Cell 3 — API Keys Configuration

In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║              SET YOUR API KEYS HERE                      ║
# ║  Groq  → https://console.groq.com/keys                  ║
# ║  Serper → https://serper.dev/api-key                    ║
# ╚══════════════════════════════════════════════════════════╝

GROQ_API_KEY   = "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"     # 🔑 Replace with your key
SERPER_API_KEY = "34cf9bb4a6965f4ad9bcce86f48f5d693048ac09"   # 🔑 Replace with your key

os.environ["GROQ_API_KEY"]   = GROQ_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

print('✅ API keys set!')

✅ API keys set!


## 🤖 Cell 4 — Initialize LLM (Groq llama-3.1-8b-instant)

In [4]:
# ╔══════════════════════════════════════════════════════════╗
# ║      INITIALIZE GROQ LLM: llama-3.1-8b-instant          ║
# ╚══════════════════════════════════════════════════════════╝

MODEL_NAME = "llama-3.1-8b-instant"

llm = ChatGroq(
    model=MODEL_NAME,
    groq_api_key=GROQ_API_KEY,
    temperature=0.7,
    max_tokens=2048,
)

# Quick test
test_response = llm.invoke([HumanMessage(content="Say: AI Tutor ready!")])
print(f'✅ LLM initialized: {MODEL_NAME}')
print(f'   Test response: {test_response.content}')

✅ LLM initialized: llama-3.1-8b-instant
   Test response: Welcome to the AI Tutor. I'm here to help you with any questions or topics you'd like to explore. What subject or topic would you like to focus on today?


## 🔍 Cell 5 — Web Search Tool (Serper API)

In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║          WEB SEARCH TOOL — Serper API                    ║
# ╚══════════════════════════════════════════════════════════╝

def web_search(query: str, num_results: int = 3) -> str:
    """Search the web using Serper API and return formatted results."""
    try:
        url = "https://google.serper.dev/search"
        headers = {
            "X-API-KEY": SERPER_API_KEY,
            "Content-Type": "application/json"
        }
        payload = {"q": query, "num": num_results}
        response = requests.post(url, json=payload, headers=headers, timeout=10)
        data = response.json()

        results = []
        if "organic" in data:
            for item in data["organic"][:num_results]:
                title   = item.get("title", "")
                snippet = item.get("snippet", "")
                link    = item.get("link", "")
                results.append(f"📌 **{title}**\n   {snippet}\n   🔗 {link}")

        # Also check knowledge graph
        if "knowledgeGraph" in data:
            kg = data["knowledgeGraph"]
            if "description" in kg:
                results.insert(0, f"📖 **Summary:** {kg['description']}")

        return "\n\n".join(results) if results else "No search results found."
    except Exception as e:
        return f"Search unavailable: {str(e)}"

print('✅ Web search tool ready!')
print('   Test: Searching for "machine learning basics"...')
# Uncomment to test:
# print(web_search('machine learning basics', 2))

✅ Web search tool ready!
   Test: Searching for "machine learning basics"...


## 🗂️ Cell 6 — LangGraph State Definition

In [6]:
# ╔══════════════════════════════════════════════════════════╗
# ║         LANGGRAPH STATE — Shared Memory Across Nodes     ║
# ╚══════════════════════════════════════════════════════════╝

class TutorState(TypedDict):
    # ── Message History ──
    messages: Annotated[List, add_messages]

    # ── Learner Profile ──
    learner_name    : str
    learner_level   : str     # beginner / intermediate / advanced
    subject         : str
    topic           : str
    learning_style  : str     # visual / auditory / reading / kinesthetic
    use_case        : str     # school / edtech / corporate

    # ── Curriculum ──
    lesson_plan     : str
    current_step    : int

    # ── Assessment ──
    quiz_questions  : List[dict]
    quiz_score      : int
    total_questions : int
    performance_history: List[dict]

    # ── Agent Control ──
    next_action     : str     # teach / quiz / feedback / search / adapt / doubt
    search_results  : str
    response        : str
    session_id      : str

print('✅ TutorState schema defined!')
print('   Fields:', list(TutorState.__annotations__.keys()))

✅ TutorState schema defined!
   Fields: ['messages', 'learner_name', 'learner_level', 'subject', 'topic', 'learning_style', 'use_case', 'lesson_plan', 'current_step', 'quiz_questions', 'quiz_score', 'total_questions', 'performance_history', 'next_action', 'search_results', 'response', 'session_id']


## 🧠 Cell 7 — LangGraph Agent Node Functions

In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║           LANGGRAPH NODE FUNCTIONS                       ║
# ╚══════════════════════════════════════════════════════════╝

# ── NODE 1: Learner Profiler ──────────────────────────────
def learner_profiler_node(state: TutorState) -> TutorState:
    """Assesses learner level and creates personalized learning profile."""
    system_prompt = f"""You are an expert AI Tutor and educational psychologist.
Learner Profile:
  Name          : {state['learner_name']}
  Level         : {state['learner_level']}
  Subject       : {state['subject']}
  Topic         : {state['topic']}
  Learning Style: {state['learning_style']}
  Context       : {state['use_case']}

Create a warm, personalized learner assessment. Include:
1. Learning profile summary (2-3 sentences)
2. Key prerequisite concepts they should know
3. Potential challenges for their level
4. Recommended teaching strategy
Use emojis to make it engaging."""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content="Analyze my profile and prepare my learning journey.")
    ]
    response = llm.invoke(messages)
    return {**state, "response": response.content, "next_action": "plan"}


# ── NODE 2: Lesson Planner ────────────────────────────────
def lesson_planner_node(state: TutorState) -> TutorState:
    """Creates a comprehensive, adaptive lesson plan."""
    system_prompt = f"""You are a master curriculum designer and instructional designer.
Create a structured, adaptive lesson plan:

Learner: {state['learner_name']} | Level: {state['learner_level']}
Subject: {state['subject']} | Topic: {state['topic']}
Style: {state['learning_style']} | Context: {state['use_case']}

Format exactly as follows:

# 📚 LESSON PLAN: {state['topic']}
**Learner:** {state['learner_name']} | **Level:** {state['learner_level'].capitalize()}
**Duration:** [estimated time]

## 🎯 Learning Objectives
By the end of this lesson, you will be able to:
1. [objective]
2. [objective]
3. [objective]

## 📋 Lesson Modules
**Module 1: [name]** ⏱️ [time]
- [key point]
- [key point]

**Module 2: [name]** ⏱️ [time]
- [key point]

**Module 3: [name]** ⏱️ [time]
- [key point]

**Module 4: [name]** ⏱️ [time]
- [key point]

## ✅ Assessment Strategy
[description — include quiz types, project, or case study]

## 🚀 Next Steps After Completion
[what to study next based on level]"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Create my lesson plan for {state['topic']}")
    ]
    response = llm.invoke(messages)
    return {**state, "lesson_plan": response.content, "response": response.content, "next_action": "teach"}


# ── NODE 3: Teaching Node ─────────────────────────────────
def teaching_node(state: TutorState) -> TutorState:
    """Delivers personalized, adaptive teaching content."""
    last_message = (
        state["messages"][-1].content
        if state["messages"]
        else f"Teach me about {state['topic']}"
    )

    style_instructions = {
        "visual"     : "Use ASCII diagrams, tables, flowcharts, and visual representations in text.",
        "reading"    : "Use structured headings, numbered lists, and well-organized paragraphs.",
        "kinesthetic": "Include hands-on exercises, practice problems, and real-world examples.",
        "auditory"   : "Use conversational tone, mnemonics, analogies, and storytelling.",
    }
    style_hint = style_instructions.get(state["learning_style"], "")

    system_prompt = f"""You are an expert {state['subject']} tutor.
Learner: {state['learner_name']} | Level: {state['learner_level']} | Style: {state['learning_style']}
Topic: {state['topic']} | Context: {state['use_case']}
Style Instruction: {style_hint}

Web Context (if relevant): {state.get('search_results', 'N/A')}

Provide a rich, engaging explanation with:
✨ **Core Concept** — Clear breakdown at {state['learner_level']} level
💡 **Real-World Example** — Practical application
🔗 **Connection** — Link to prior knowledge
📝 **Key Takeaway** — 3 bullet summary
❓ **Check Understanding** — One thoughtful question at the end"""

    messages = [SystemMessage(content=system_prompt), HumanMessage(content=last_message)]
    response = llm.invoke(messages)
    return {**state, "response": response.content, "next_action": "teach"}


# ── NODE 4: Quiz Generator ────────────────────────────────
def quiz_generator_node(state: TutorState) -> TutorState:
    """Generates adaptive, level-appropriate quiz questions."""
    system_prompt = f"""You are an expert assessment designer.
Generate 5 multiple-choice questions on '{state['topic']}' for a {state['learner_level']} learner.
Subject: {state['subject']}

Return ONLY a valid JSON array. No markdown, no explanation, just JSON:
[
  {{
    "id": 1,
    "question": "Question text?",
    "options": ["A) option1", "B) option2", "C) option3", "D) option4"],
    "correct": "A",
    "explanation": "Why this answer is correct.",
    "difficulty": "easy"
  }}
]"""

    messages = [SystemMessage(content=system_prompt), HumanMessage(content="Generate quiz now.")]
    response = llm.invoke(messages)

    try:
        clean = response.content.strip()
        if "```" in clean:
            clean = re.sub(r'```(?:json)?', '', clean).strip().strip('`')
        # Find first '[' to last ']'
        start, end = clean.find('['), clean.rfind(']')
        if start != -1 and end != -1:
            clean = clean[start:end+1]
        questions = json.loads(clean)
    except Exception as e:
        questions = [
            {
                "id": i+1,
                "question": f"Concept check {i+1} on {state['topic']}?",
                "options": ["A) Option A", "B) Option B", "C) Option C", "D) Option D"],
                "correct": "A",
                "explanation": "Review the lesson material.",
                "difficulty": "medium"
            } for i in range(5)
        ]

    # Build readable quiz text
    diff_emoji = {"easy": "🟢", "medium": "🟡", "hard": "🔴"}
    quiz_text  = f"## 📝 Quiz: {state['topic']}\n"
    quiz_text += f"*Level: {state['learner_level'].capitalize()} | {len(questions)} Questions*\n\n"
    quiz_text += "---\n"

    for q in questions[:5]:
        diff  = q.get('difficulty', 'medium')
        emoji = diff_emoji.get(diff, '🟡')
        quiz_text += f"\n**Q{q['id']}. {q['question']}** {emoji}\n\n"
        for opt in q.get('options', []):
            quiz_text += f"  {opt}\n"
        quiz_text += "\n"

    quiz_text += "---\n"
    quiz_text += "*Enter your answers in the Answers box below (e.g., Q1:A, Q2:B, Q3:C)*"

    return {**state, "quiz_questions": questions, "response": quiz_text, "next_action": "quiz"}


# ── NODE 5: Feedback Engine ───────────────────────────────
def feedback_node(state: TutorState) -> TutorState:
    """Evaluates quiz performance and provides constructive feedback."""
    last_message = state["messages"][-1].content if state["messages"] else "No answers provided"
    questions    = state.get("quiz_questions", [])

    system_prompt = f"""You are a warm, supportive tutor evaluating a quiz.
Learner: {state['learner_name']} ({state['learner_level']} level)
Topic: {state['topic']}
Learner's answers: {last_message}
Quiz questions with correct answers: {json.dumps(questions)}

Provide a detailed, encouraging feedback report:
## 🎯 Quiz Results
**Score:** [X/5] — [percentage]%

## ✅ Correct Answers
[List what they got right with brief praise]

## 📖 Learning Opportunities
[For wrong answers: explain the correct concept gently]

## 💪 Overall Feedback
[Warm encouragement + specific next steps]

## 📈 Recommended Actions
1. [action]
2. [action]
3. [action]"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Evaluate: {last_message}")
    ]
    response = llm.invoke(messages)

    # Estimate score from LLM response
    score_match = re.search(r'(\d+)/5', response.content)
    score = int(score_match.group(1)) * 20 if score_match else random.randint(60, 90)

    perf = {
        "timestamp": datetime.now().isoformat(),
        "score"    : score,
        "topic"    : state["topic"],
        "level"    : state["learner_level"]
    }
    history = state.get("performance_history", []) + [perf]

    return {
        **state,
        "response"           : response.content,
        "quiz_score"         : score,
        "performance_history": history,
        "next_action"        : "adapt"
    }


# ── NODE 6: Adaptive Engine ───────────────────────────────
def adaptive_engine_node(state: TutorState) -> TutorState:
    """Analyzes performance and adapts the learning path dynamically."""
    history  = state.get("performance_history", [])
    avg_score = sum(p["score"] for p in history) / len(history) if history else 75

    # Auto-level adjustment
    new_level = state["learner_level"]
    if avg_score >= 85 and state["learner_level"] == "beginner":
        new_level = "intermediate"
        level_msg = "🎉 Promoted to Intermediate!"
    elif avg_score >= 85 and state["learner_level"] == "intermediate":
        new_level = "advanced"
        level_msg = "🚀 Promoted to Advanced!"
    elif avg_score < 55 and state["learner_level"] == "advanced":
        new_level = "intermediate"
        level_msg = "📚 Adjusting to Intermediate for better consolidation."
    elif avg_score < 50 and state["learner_level"] == "intermediate":
        new_level = "beginner"
        level_msg = "📖 Revisiting Beginner fundamentals."
    else:
        level_msg = f"✅ Continuing at {new_level.capitalize()} level."

    system_prompt = f"""You are an adaptive learning AI.
Learner: {state['learner_name']}
Current Level: {state['learner_level']} → New Level: {new_level}
Average Score: {avg_score:.1f}%
Performance History: {json.dumps(history)}
Level Change Message: {level_msg}

Generate an adaptive learning report:
## 📊 Performance Analysis
[3-4 sentence analysis of performance trends]

## 🎯 Level Adjustment
{level_msg}
[Explanation of why and what changes]

## 📚 Recommended Next Topics
1. [topic] — [why]
2. [topic] — [why]
3. [topic] — [why]

## ⚡ Your Action Plan
• [specific action]
• [specific action]
• [specific action]

## 🌟 Motivation
[Personalized encouragement — use {state['learner_name']}'s name]"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content="Adapt my learning path based on my performance.")
    ]
    response = llm.invoke(messages)

    return {
        **state,
        "response"      : response.content,
        "learner_level" : new_level,
        "next_action"   : "teach"
    }


# ── NODE 7: Web Search ────────────────────────────────────
def web_search_node(state: TutorState) -> TutorState:
    """Fetches current, relevant information from the web."""
    last_message = state["messages"][-1].content if state["messages"] else state["topic"]
    query        = f"{state['topic']} {last_message} educational tutorial"
    results      = web_search(query)
    return {**state, "search_results": results, "next_action": "doubt"}


# ── NODE 8: Doubt Resolver ────────────────────────────────
def doubt_resolver_node(state: TutorState) -> TutorState:
    """Resolves learner doubts with detailed, contextual explanations."""
    last_message = state["messages"][-1].content if state["messages"] else ""

    system_prompt = f"""You are an infinitely patient, expert tutor.
Subject: {state['subject']} | Topic: {state['topic']}
Learner Level: {state['learner_level']} | Style: {state['learning_style']}
Web Context (if available): {state.get('search_results', 'Not available')}

Answer the learner's question thoroughly:
1. 🤝 Acknowledge their question warmly
2. 💡 Provide a clear, {state['learner_level']}-appropriate explanation
3. 🔍 Use analogies and concrete examples
4. 🔗 Connect to concepts they already know
5. 📝 Summarize in 2-3 bullet points
6. ❓ End with a follow-up question to reinforce understanding"""

    messages = [SystemMessage(content=system_prompt), HumanMessage(content=last_message)]
    response  = llm.invoke(messages)
    return {**state, "response": response.content, "next_action": "teach"}


print('✅ All 8 agent nodes defined!')
print('   Nodes: Profiler | Lesson Planner | Teacher | Quiz | Feedback | Adaptive | Search | Doubt')

✅ All 8 agent nodes defined!
   Nodes: Profiler | Lesson Planner | Teacher | Quiz | Feedback | Adaptive | Search | Doubt


## 🕸️ Cell 8 — Build LangGraph Workflow

In [8]:
# ╔══════════════════════════════════════════════════════════╗
# ║       BUILD & COMPILE LANGGRAPH AGENT WORKFLOW           ║
# ╚══════════════════════════════════════════════════════════╝

def build_tutor_graph():
    graph = StateGraph(TutorState)

    # ── Register all nodes ──
    graph.add_node("profiler",       learner_profiler_node)
    graph.add_node("lesson_planner", lesson_planner_node)
    graph.add_node("teacher",        teaching_node)
    graph.add_node("quiz_generator", quiz_generator_node)
    graph.add_node("feedback",       feedback_node)
    graph.add_node("adaptive_engine",adaptive_engine_node)
    graph.add_node("web_search",     web_search_node)
    graph.add_node("doubt_resolver", doubt_resolver_node)

    # ── Define edges ──
    graph.set_entry_point("profiler")

    # Profiler always goes to Lesson Planner
    graph.add_edge("profiler",        "lesson_planner")
    graph.add_edge("lesson_planner",  "teacher")
    graph.add_edge("teacher",          END)
    graph.add_edge("quiz_generator",   END)
    graph.add_edge("feedback",         END)
    graph.add_edge("adaptive_engine",  END)
    graph.add_edge("web_search",       "doubt_resolver")  # Search → then answer
    graph.add_edge("doubt_resolver",   END)

    return graph.compile()


tutor_graph = build_tutor_graph()
print('✅ LangGraph Tutor Agent compiled successfully!')
print('   Entry: profiler → lesson_planner → teacher → [END]')
print('   Branches: quiz_generator | feedback | adaptive_engine | web_search → doubt_resolver')

✅ LangGraph Tutor Agent compiled successfully!
   Entry: profiler → lesson_planner → teacher → [END]
   Branches: quiz_generator | feedback | adaptive_engine | web_search → doubt_resolver


## 💼 Cell 9 — Session Management & Helper Functions

In [9]:
# ╔══════════════════════════════════════════════════════════╗
# ║        SESSION MANAGEMENT & CORE ACTION FUNCTIONS        ║
# ╚══════════════════════════════════════════════════════════╝

# ── Global session store ──
session_states: dict = {}

def get_or_create_session(
    session_id: str, learner_name: str, level: str,
    subject: str, topic: str, style: str, use_case: str
) -> TutorState:
    """Get existing or create a new learning session."""
    if session_id not in session_states:
        session_states[session_id] = TutorState(
            messages=[], learner_name=learner_name, learner_level=level,
            subject=subject, topic=topic, learning_style=style, use_case=use_case,
            lesson_plan="", current_step=0, quiz_questions=[], quiz_score=0,
            total_questions=5, performance_history=[], next_action="plan",
            search_results="", response="", session_id=session_id
        )
    else:
        session_states[session_id].update({
            "learner_name": learner_name, "learner_level": level,
            "subject": subject, "topic": topic,
            "learning_style": style, "use_case": use_case
        })
    return session_states[session_id]


# ── Action: Start Session ─────────────────────────────────
def start_session(learner_name, level, subject, topic, style, use_case):
    """Initialize learner profile and run the LangGraph agent."""
    if not learner_name.strip():
        return "", "⚠️ Please enter your name to begin.", []

    session_id = f"{learner_name.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
    state = get_or_create_session(session_id, learner_name, level, subject, topic, style, use_case)
    state["next_action"] = "plan"
    state["messages"]   = [HumanMessage(content=f"Begin my learning journey on {topic}")]

    result = tutor_graph.invoke(state)
    session_states[session_id] = result

    plan    = result.get("lesson_plan") or result.get("response", "Session started!")
    history = [(f"🚀 Session started for {learner_name}", plan)]
    return session_id, plan, history


# ── Action: Ask Question ──────────────────────────────────
def ask_question(user_input, session_id, chat_history, use_web_search):
    """Handle learner's question with intent detection."""
    if not session_id or session_id not in session_states:
        return chat_history, "⚠️ Please start a session first (Tab 1)."
    if not user_input.strip():
        return chat_history, ""

    state = session_states[session_id]
    state["messages"] = state["messages"] + [HumanMessage(content=user_input)]
    low = user_input.lower()

    # ── Intent Detection ──
    if any(w in low for w in ["quiz", "test me", "assessment", "evaluate me", "challenge"]):
        result = quiz_generator_node(state)
    elif any(w in low for w in ["feedback", "score", "how did i do", "check my answers"]):
        result = feedback_node(state)
    elif any(w in low for w in ["adapt", "next step", "progress", "what should i study", "learning path"]):
        result = adaptive_engine_node(state)
    elif use_web_search or any(w in low for w in ["latest", "recent", "current", "2024", "2025", "news"]):
        state = web_search_node(state)
        result = doubt_resolver_node(state)
    else:
        result = doubt_resolver_node(state)

    session_states[session_id] = result
    updated_history = list(chat_history) + [(user_input, result["response"])]
    return updated_history, ""


# ── Action: Generate Lesson Plan ──────────────────────────
def generate_lesson_plan(session_id):
    if not session_id or session_id not in session_states:
        return "⚠️ Start a session first."
    state = session_states[session_id]
    result = lesson_planner_node(state)
    session_states[session_id] = result
    return result["lesson_plan"]


# ── Action: Start Quiz ────────────────────────────────────
def start_quiz(session_id, chat_history):
    if not session_id or session_id not in session_states:
        return chat_history, "⚠️ Start a session first."
    state = session_states[session_id]
    state["messages"] = state["messages"] + [HumanMessage(content="Start quiz")]
    result = quiz_generator_node(state)
    session_states[session_id] = result
    return list(chat_history) + [("📝 Start quiz", result["response"])], ""


# ── Action: Adaptive Learning ─────────────────────────────
def adapt_learning(session_id, chat_history):
    if not session_id or session_id not in session_states:
        return chat_history, "⚠️ Start a session first."
    state = session_states[session_id]
    result = adaptive_engine_node(state)
    session_states[session_id] = result
    return list(chat_history) + [("🔄 Adapt my learning path", result["response"])], ""


# ── Action: Performance Report ────────────────────────────
def get_performance_report(session_id):
    if not session_id or session_id not in session_states:
        return "⚠️ Start a session first."
    state   = session_states[session_id]
    history = state.get("performance_history", [])
    if not history:
        return "📊 No quiz data yet. Take a quiz first!"

    avg  = sum(p["score"] for p in history) / len(history)
    best = max(p["score"] for p in history)
    last = history[-1]["score"]

    report  = f"## 📊 Performance Report — {state['learner_name']}\n\n"
    report += f"| Metric | Value |\n|--------|-------|\n"
    report += f"| 👤 Current Level | {state['learner_level'].capitalize()} |\n"
    report += f"| 📚 Subject | {state['subject']} |\n"
    report += f"| 🎯 Topic | {state['topic']} |\n"
    report += f"| 📈 Average Score | **{avg:.1f}%** |\n"
    report += f"| 🏆 Best Score | {best}% |\n"
    report += f"| 📋 Last Score | {last}% |\n"
    report += f"| 🔢 Quizzes Taken | {len(history)} |\n\n"
    report += "### 📈 Quiz History\n"

    for i, p in enumerate(history, 1):
        bar   = "█" * (p["score"] // 10) + "░" * (10 - p["score"] // 10)
        emoji = "🌟" if p["score"] >= 80 else ("✅" if p["score"] >= 60 else "⚠️")
        report += f"{emoji} Attempt {i}: [{bar}] **{p['score']}%** — {p['topic']}\n"

    if avg >= 80:
        report += "\n### 🚀 Assessment: Ready to advance to the next level!"
    elif avg >= 60:
        report += "\n### 💪 Assessment: Good progress! Keep practicing consistently."
    else:
        report += "\n### 📖 Assessment: Review recommended. Let's revisit the fundamentals."
    return report


# ── Action: Search Resources ──────────────────────────────
def search_resources(session_id):
    if not session_id or session_id not in session_states:
        return "⚠️ Start a session first."
    state   = session_states[session_id]
    results = web_search(f"best resources learn {state['topic']} {state['subject']} {state['learner_level']}")
    return f"## 🔍 Resources: {state['topic']}\n\n{results}"


print('✅ All session management functions ready!')

✅ All session management functions ready!


## 🎨 Cell 10 — Gradio UI Configuration (CSS + Data)

In [10]:
# ╔══════════════════════════════════════════════════════════╗
# ║        GRADIO UI — CSS STYLES & DATA                     ║
# ╚══════════════════════════════════════════════════════════╝

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;700;800&family=DM+Serif+Display&display=swap');

:root {
    --primary: #4F46E5; --primary-light: #818CF8;
    --secondary: #06B6D4; --accent: #F59E0B;
    --success: #10B981; --danger: #EF4444;
    --bg-main: #F0F4FF; --bg-card: #FFFFFF;
    --bg-panel: #EEF2FF; --text-primary: #1E1B4B;
    --text-secondary: #6B7280; --border: #C7D2FE;
    --shadow: 0 4px 24px rgba(79,70,229,0.10);
    --radius: 16px;
}

body, .gradio-container {
    background: var(--bg-main) !important;
    font-family: 'Nunito', sans-serif !important;
    color: var(--text-primary) !important;
}

.block, .gr-block {
    background: var(--bg-card) !important;
    border: 1.5px solid var(--border) !important;
    border-radius: var(--radius) !important;
    box-shadow: var(--shadow) !important;
}

.tab-nav button {
    border-radius: 10px !important; font-weight: 700 !important;
    font-family: 'Nunito', sans-serif !important; font-size: 14px !important;
    padding: 10px 20px !important; transition: all 0.2s !important;
    color: var(--text-secondary) !important; border: none !important;
}
.tab-nav button.selected {
    background: linear-gradient(135deg, var(--primary), #7C3AED) !important;
    color: white !important; box-shadow: 0 4px 12px rgba(79,70,229,0.3) !important;
}

input, textarea, select {
    background: var(--bg-panel) !important;
    border: 1.5px solid var(--border) !important;
    border-radius: 10px !important;
    font-family: 'Nunito', sans-serif !important;
    color: var(--text-primary) !important;
}
input:focus, textarea:focus {
    border-color: var(--primary) !important;
    box-shadow: 0 0 0 3px rgba(79,70,229,0.15) !important;
    outline: none !important;
}

button.primary {
    background: linear-gradient(135deg, #4F46E5, #7C3AED) !important;
    color: white !important; border: none !important;
    border-radius: 12px !important; font-weight: 700 !important;
    font-family: 'Nunito', sans-serif !important;
    box-shadow: 0 4px 14px rgba(79,70,229,0.35) !important;
    transition: all 0.2s ease !important;
}
button.primary:hover { transform: translateY(-2px) !important; }

button.secondary {
    background: var(--bg-panel) !important;
    color: var(--primary) !important;
    border: 2px solid var(--primary) !important;
    border-radius: 12px !important; font-weight: 700 !important;
    font-family: 'Nunito', sans-serif !important;
}

.chatbot .message.user {
    background: linear-gradient(135deg, #4F46E5, #7C3AED) !important;
    color: white !important;
    border-radius: 16px 16px 4px 16px !important;
}
.chatbot .message.bot {
    background: var(--bg-panel) !important;
    border-radius: 16px 16px 16px 4px !important;
    border: 1.5px solid var(--border) !important;
}

label, .label-wrap span {
    font-weight: 700 !important; color: var(--text-primary) !important;
    font-family: 'Nunito', sans-serif !important; font-size: 13px !important;
}
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-thumb { background: var(--primary-light); border-radius: 10px; }
"""

HEADER_HTML = """
<div style="background:linear-gradient(135deg,#4F46E5,#7C3AED 40%,#06B6D4);
            border-radius:16px; padding:32px 40px; margin-bottom:8px;
            box-shadow:0 8px 32px rgba(79,70,229,.25); text-align:center;">
  <div style="font-size:52px;margin-bottom:8px;">🎓</div>
  <h1 style="font-family:'DM Serif Display',serif;color:white;font-size:38px;
             margin:0 0 8px;letter-spacing:-0.5px;">AI Tutor Agent</h1>
  <p style="color:rgba(255,255,255,.85);font-size:15px;margin:0;
            font-family:'Nunito',sans-serif;font-weight:600;">
    Powered by LangGraph &middot; Groq llama-3.1-8b-instant &middot; Serper Web Search
  </p>
  <div style="display:flex;gap:12px;justify-content:center;margin-top:16px;flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,.2);color:white;padding:6px 16px;
                 border-radius:20px;font-size:12px;font-weight:700;">🏫 Schools</span>
    <span style="background:rgba(255,255,255,.2);color:white;padding:6px 16px;
                 border-radius:20px;font-size:12px;font-weight:700;">💻 EdTech</span>
    <span style="background:rgba(255,255,255,.2);color:white;padding:6px 16px;
                 border-radius:20px;font-size:12px;font-weight:700;">🏢 Corporate</span>
    <span style="background:rgba(255,255,255,.2);color:white;padding:6px 16px;
                 border-radius:20px;font-size:12px;font-weight:700;">🤖 Adaptive AI</span>
  </div>
</div>
"""

SUBJECTS = [
    "Mathematics", "Physics", "Chemistry", "Biology",
    "Computer Science", "Data Science & ML", "History",
    "Economics", "Literature", "Programming",
    "Business Strategy", "Statistics", "Cybersecurity",
    "Cloud Computing", "Psychology", "Finance"
]

TOPICS_MAP = {
    "Mathematics"        : ["Algebra", "Calculus", "Linear Algebra", "Probability", "Geometry", "Number Theory"],
    "Computer Science"   : ["Data Structures", "Algorithms", "OOP", "Databases", "Networks", "OS Concepts"],
    "Data Science & ML"  : ["Machine Learning", "Deep Learning", "Neural Networks", "NLP", "Computer Vision", "Data Visualization"],
    "Physics"            : ["Mechanics", "Thermodynamics", "Electromagnetism", "Quantum Mechanics", "Optics"],
    "Chemistry"          : ["Organic Chemistry", "Inorganic Chemistry", "Chemical Bonds", "Thermochemistry"],
    "Business Strategy"  : ["Porter's Five Forces", "SWOT Analysis", "Growth Strategy", "Market Analysis", "Leadership"],
    "Programming"        : ["Python Basics", "Web Development", "APIs & REST", "Version Control", "Debugging", "Design Patterns"],
    "Cybersecurity"      : ["Network Security", "Cryptography", "Penetration Testing", "OWASP Top 10", "Zero Trust"],
    "Cloud Computing"    : ["AWS Fundamentals", "Azure Services", "GCP Basics", "Kubernetes", "Serverless Architecture"],
    "Finance"            : ["Financial Modeling", "Investment Analysis", "Risk Management", "Derivatives", "Valuation"],
}

def update_topics(subject):
    topics = TOPICS_MAP.get(subject, [f"{subject} Fundamentals", f"Advanced {subject}", f"{subject} Applications"])
    return gr.Dropdown(choices=topics, value=topics[0] if topics else "")

print('✅ CSS, HTML, and UI data configured!')

✅ CSS, HTML, and UI data configured!


## 🎨 Cell 11 — Build Professional Gradio UI

In [11]:
# ╔══════════════════════════════════════════════════════════╗
# ║       PROFESSIONAL GRADIO UI — 4 INTERACTIVE TABS        ║
# ╚══════════════════════════════════════════════════════════╝

def build_gradio_app():
    with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), title="🎓 AI Tutor Agent") as app:

        # ── Header ──
        gr.HTML(HEADER_HTML)

        # ── Hidden States ──
        session_id_state   = gr.State("")
        chat_history_state = gr.State([])

        with gr.Tabs():

            # ═══════════════════════════════════════════
            # TAB 1 — Learner Setup
            # ═══════════════════════════════════════════
            with gr.Tab("👤 Learner Setup"):
                with gr.Row():
                    with gr.Column(scale=2):
                        gr.Markdown("### 📋 Configure Your Learning Profile")
                        with gr.Row():
                            learner_name = gr.Textbox(
                                label="👋 Your Name",
                                placeholder="Enter your name...",
                                scale=2
                            )
                            use_case = gr.Dropdown(
                                label="🏛️ Learning Context",
                                choices=["school", "edtech", "corporate"],
                                value="edtech", scale=1
                            )
                        with gr.Row():
                            learner_level = gr.Dropdown(
                                label="📊 Knowledge Level",
                                choices=["beginner", "intermediate", "advanced"],
                                value="beginner"
                            )
                            learning_style = gr.Dropdown(
                                label="🎨 Learning Style",
                                choices=["visual", "reading", "kinesthetic", "auditory"],
                                value="visual"
                            )
                        with gr.Row():
                            subject = gr.Dropdown(
                                label="📚 Subject",
                                choices=SUBJECTS,
                                value="Data Science & ML"
                            )
                            topic = gr.Dropdown(
                                label="🎯 Topic",
                                choices=TOPICS_MAP["Data Science & ML"],
                                value="Machine Learning"
                            )
                        subject.change(fn=update_topics, inputs=subject, outputs=topic)

                        with gr.Row():
                            start_btn = gr.Button("🚀 Start Learning Session", variant="primary", size="lg")
                            plan_btn  = gr.Button("📋 Regenerate Lesson Plan", variant="secondary")

                        session_display = gr.Textbox(
                            label="🔑 Session ID", interactive=False,
                            placeholder="Session ID will appear here after starting..."
                        )

                    with gr.Column(scale=3):
                        gr.Markdown("### 📖 Personalized Lesson Plan")
                        lesson_plan_output = gr.Markdown(
                            value="*Click 'Start Learning Session' to generate your personalized lesson plan...*"
                        )

                with gr.Accordion("ℹ️ About This AI Tutor Agent", open=False):
                    gr.Markdown("""
                    ## 🤖 LangGraph Multi-Agent Architecture

                    | 🟣 Node | 🎯 Role |
                    |---------|--------|
                    | 👤 **Learner Profiler** | Builds personalized learning profile |
                    | 📋 **Lesson Planner** | Creates adaptive curriculum |
                    | 🎓 **Teaching Node** | Delivers style-adapted content |
                    | 📝 **Quiz Generator** | Creates level-appropriate assessments |
                    | ✅ **Feedback Engine** | Evaluates with encouragement |
                    | 📈 **Adaptive Engine** | Auto-adjusts difficulty & level |
                    | 🔍 **Web Search** | Fetches latest info via Serper API |
                    | ❓ **Doubt Resolver** | Answers questions with context |

                    **Tech Stack:** LangGraph · Groq `llama-3.1-8b-instant` · Serper API · Gradio
                    """)

            # ═══════════════════════════════════════════
            # TAB 2 — Chat with Tutor
            # ═══════════════════════════════════════════
            with gr.Tab("💬 Chat with Tutor"):
                with gr.Row():
                    with gr.Column(scale=3):
                        chatbot = gr.Chatbot(
                            label="🎓 Your AI Tutor",
                            height=480,
                            bubble_full_width=False,
                            avatar_images=("👤", "🤖")
                        )
                        with gr.Row():
                            user_input = gr.Textbox(
                                label="",
                                placeholder="💬 Ask anything... e.g. 'Explain backpropagation' or 'Give me a quiz'",
                                lines=2, scale=5
                            )
                            send_btn = gr.Button("Send ➤", variant="primary", scale=1)
                        with gr.Row():
                            use_web   = gr.Checkbox(label="🌐 Enable Web Search", value=False)
                            clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

                    with gr.Column(scale=1):
                        gr.Markdown("### ⚡ Quick Actions")
                        quiz_btn_chat  = gr.Button("📝 Take a Quiz",       variant="primary",   size="sm")
                        adapt_btn_chat = gr.Button("🔄 Adapt My Path",     variant="secondary", size="sm")

                        gr.Markdown("### 💡 Starter Questions")
                        q1 = gr.Button("🔍 Explain this concept",          size="sm")
                        q2 = gr.Button("📊 Give me a real-world example",  size="sm")
                        q3 = gr.Button("⚠️ What are common mistakes?",     size="sm")
                        q4 = gr.Button("🔗 Connect to real world",         size="sm")
                        q5 = gr.Button("📈 What should I study next?",     size="sm")
                        q6 = gr.Button("🧪 Give me a practice exercise",   size="sm")

                        gr.Markdown("### 📊 Session Info")
                        stats_box = gr.Markdown(
                            "*Start a session to see info*",
                            elem_classes=["info-box"]
                        )

            # ═══════════════════════════════════════════
            # TAB 3 — Quiz Center
            # ═══════════════════════════════════════════
            with gr.Tab("📝 Quiz Center"):
                with gr.Row():
                    with gr.Column(scale=2):
                        gr.Markdown("### 🎯 Adaptive Quiz Assessment")
                        with gr.Row():
                            quiz_difficulty = gr.Dropdown(
                                label="📊 Quiz Difficulty",
                                choices=["Auto (Based on Level)", "Easy", "Medium", "Hard", "Mixed"],
                                value="Auto (Based on Level)"
                            )
                            quiz_num = gr.Dropdown(
                                label="🔢 Number of Questions",
                                choices=["3", "5", "10"],
                                value="5"
                            )
                        start_quiz_btn = gr.Button("🚀 Generate New Quiz", variant="primary", size="lg")

                        quiz_output = gr.Markdown(
                            "*Click 'Generate New Quiz' to begin your assessment...*"
                        )

                        gr.Markdown("### ✍️ Submit Your Answers")
                        answers_input = gr.Textbox(
                            label="Answers (format: Q1:A, Q2:B, Q3:C, Q4:D, Q5:A)",
                            placeholder="Q1:A, Q2:C, Q3:B, Q4:D, Q5:A",
                            lines=2
                        )
                        submit_btn = gr.Button("✅ Submit & Get Feedback", variant="primary")
                        quiz_feedback = gr.Markdown("*Submit answers for detailed feedback...*")

                    with gr.Column(scale=1):
                        gr.Markdown("### 📊 Performance Analytics")
                        perf_btn = gr.Button("📈 View Full Report", variant="primary")
                        perf_output = gr.Markdown(
                            "*Take quizzes to build your performance report...*"
                        )

            # ═══════════════════════════════════════════
            # TAB 4 — Adaptive Learning Path
            # ═══════════════════════════════════════════
            with gr.Tab("🗺️ Learning Path"):
                with gr.Row():
                    with gr.Column(scale=2):
                        gr.Markdown("### 📈 Adaptive Learning Journey")
                        with gr.Row():
                            adapt_btn_path  = gr.Button("🔄 Analyze & Adapt Path", variant="primary", size="lg")
                            search_res_btn  = gr.Button("🔍 Search Latest Resources", variant="secondary", size="lg")
                        path_output = gr.Markdown(
                            "*Complete quizzes to unlock your adaptive learning path...*"
                        )

                        gr.Markdown("### 🌐 Web Resources")
                        search_output = gr.Markdown("*Click 'Search Latest Resources' to find current content...*")

                    with gr.Column(scale=1):
                        gr.Markdown("### 🏆 Level System")
                        gr.Markdown("""
**🌱 Beginner** (Score < 60%)
Foundational concepts, basic terminology, simple examples

---
**📘 Intermediate** (Score 60–80%)
Application, problem-solving, connecting concepts

---
**🚀 Advanced** (Score > 80%)
Deep understanding, original thinking, mastery projects

---
*The AI automatically adjusts your level based on quiz performance!*
                        """)

                        gr.Markdown("### 📚 Resources")
                        resource_box = gr.Markdown("*Start learning to see recommendations*")

        # ═══════════════════════════════════════════════
        # Event Wiring
        # ═══════════════════════════════════════════════

        def start_and_update(name, level, subj, top, style, ctx):
            sid, plan, hist = start_session(name, level, subj, top, style, ctx)
            stats = f"✅ **Session Active**\n👤 {name} | 📊 {level.capitalize()}\n📚 {subj} → {top}"
            return sid, plan, hist, stats

        start_btn.click(
            fn=start_and_update,
            inputs=[learner_name, learner_level, subject, topic, learning_style, use_case],
            outputs=[session_id_state, lesson_plan_output, chat_history_state, stats_box]
        ).then(lambda s: s, inputs=session_id_state, outputs=session_display)

        plan_btn.click(fn=generate_lesson_plan, inputs=[session_id_state], outputs=[lesson_plan_output])

        def send_msg(msg, sid, hist, web):
            updated, _ = ask_question(msg, sid, hist, web)
            return updated, ""

        send_btn.click(
            fn=send_msg,
            inputs=[user_input, session_id_state, chat_history_state, use_web],
            outputs=[chatbot, user_input]
        ).then(lambda h: h, inputs=chatbot, outputs=chat_history_state)

        user_input.submit(
            fn=send_msg,
            inputs=[user_input, session_id_state, chat_history_state, use_web],
            outputs=[chatbot, user_input]
        ).then(lambda h: h, inputs=chatbot, outputs=chat_history_state)

        clear_btn.click(fn=lambda: ([], []), outputs=[chatbot, chat_history_state])

        quiz_btn_chat.click(
            fn=start_quiz,
            inputs=[session_id_state, chat_history_state],
            outputs=[chatbot, user_input]
        ).then(lambda h: h, inputs=chatbot, outputs=chat_history_state)

        adapt_btn_chat.click(
            fn=adapt_learning,
            inputs=[session_id_state, chat_history_state],
            outputs=[chatbot, user_input]
        ).then(lambda h: h, inputs=chatbot, outputs=chat_history_state)

        # Starter question buttons
        for btn, txt in [
            (q1, "Please explain this concept clearly with examples"),
            (q2, "Give me a real-world example of this topic"),
            (q3, "What are the most common mistakes beginners make?"),
            (q4, "How does this connect to real-world applications?"),
            (q5, "What should I study after this? Give me a roadmap."),
            (q6, "Give me a practice exercise I can try right now"),
        ]:
            btn.click(
                fn=lambda sid, hist, t=txt: ask_question(t, sid, hist, False),
                inputs=[session_id_state, chat_history_state],
                outputs=[chatbot, user_input]
            ).then(lambda h: h, inputs=chatbot, outputs=chat_history_state)

        # Quiz tab
        start_quiz_btn.click(
            fn=lambda sid, hist: (start_quiz(sid, hist)[0][-1][1] if start_quiz(sid, hist)[0] else ""),
            inputs=[session_id_state, chat_history_state],
            outputs=[quiz_output]
        )

        def submit_quiz_answers(answers, sid, hist):
            updated, _ = ask_question(f"My quiz answers are: {answers}", sid, hist, False)
            return updated[-1][1] if updated else "No feedback available."

        submit_btn.click(
            fn=submit_quiz_answers,
            inputs=[answers_input, session_id_state, chat_history_state],
            outputs=[quiz_feedback]
        )

        perf_btn.click(fn=get_performance_report, inputs=[session_id_state], outputs=[perf_output])

        # Learning Path tab
        adapt_btn_path.click(
            fn=lambda sid, hist: adapt_learning(sid, hist)[0][-1][1] if adapt_learning(sid, hist)[0] else "",
            inputs=[session_id_state, chat_history_state],
            outputs=[path_output]
        )
        search_res_btn.click(
            fn=search_resources,
            inputs=[session_id_state],
            outputs=[search_output]
        )

    return app


app = build_gradio_app()
print('✅ Gradio UI built successfully!')

/tmp/ipykernel_2537/690875538.py:6: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), title="🎓 AI Tutor Agent") as app:
/tmp/ipykernel_2537/690875538.py:6: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), title="🎓 AI Tutor Agent") as app:
/tmp/ipykernel_2537/690875538.py:98: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_2537/690875538.py:98: DeprecationWarning: The 'bubble_f

✅ Gradio UI built successfully!


## 🚀 Cell 12 — Launch the Application

In [12]:
# ╔══════════════════════════════════════════════════════════╗
# ║              🚀 LAUNCH AI TUTOR AGENT                    ║
# ╚══════════════════════════════════════════════════════════╝

print("\n" + "═"*58)
print("  🎓  AI TUTOR AGENT — Launching Now!")
print("═"*58)
print(f"  🤖  Model   : {MODEL_NAME} (Groq)")
print(f"  🕸️  Search  : Serper API")
print(f"  🔗  Graph   : LangGraph (8 nodes)")
print(f"  🎨  UI      : Gradio Professional")
print("─"*58)
print("  Use Cases: 🏫 Schools | 💻 EdTech | 🏢 Corporate")
print("═"*58 + "\n")

app.launch(
    share=True,          # Creates public URL — accessible from anywhere
    debug=False,
    show_error=True,
    server_name="0.0.0.0",
    server_port=7860,
    quiet=False
)


══════════════════════════════════════════════════════════
  🎓  AI TUTOR AGENT — Launching Now!
══════════════════════════════════════════════════════════
  🤖  Model   : llama-3.1-8b-instant (Groq)
  🕸️  Search  : Serper API
  🔗  Graph   : LangGraph (8 nodes)
  🎨  UI      : Gradio Professional
──────────────────────────────────────────────────────────
  Use Cases: 🏫 Schools | 💻 EdTech | 🏢 Corporate
══════════════════════════════════════════════════════════

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4fbcfc8facdf1809d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
